In [ ]:
# thư viện và dữ liệu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from sklearn.model_selection import ParameterGrid

# tải dữ liệu
# YYYY-MM-DD (ngày đầu tháng)
df = pd.read_csv('../tnbike_data.csv')
df

In [ ]:
# đổi tên biến
df = df.rename(columns={'revenue': 'y',
                        'ds':      'ds'})
df.head(1)

In [ ]:
# biến ngày
df.ds = pd.to_datetime(df.ds, format="%Y-%m-%d")
df.ds

# Ngày lễ

In [ ]:
# Tết Nguyên Đán
dates_tet = pd.to_datetime(['2025-01-29', '2026-01-17'])
tet = pd.DataFrame({"holiday": "tet_nguyen_dan",
                    "ds": dates_tet,
                    "lower_window": -7,
                    "upper_window": 7})

In [ ]:
# Ngày Thiếu nhi 1/6 (ảnh hưởng KIDBIKE)
dates_thieu_nhi = pd.to_datetime(['2025-06-01', '2026-06-01'])
thieu_nhi = pd.DataFrame({"holiday": "ngay_thieu_nhi",
                           "ds": dates_thieu_nhi,
                           "lower_window": -14,
                           "upper_window": 3})

In [ ]:
thieu_nhi

In [ ]:
# gộp các ngày lễ
holidays = pd.concat([tet, thieu_nhi])
holidays

In [ ]:
df = df.drop(["is_tet", "is_thieu_nhi"], axis=1, errors='ignore')

In [ ]:
df.head(0)

In [ ]:
m = Prophet(holidays=holidays,
            seasonality_mode='multiplicative',
            seasonality_prior_scale=10,
            holidays_prior_scale=10,
            changepoint_prior_scale=0.05)
m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
m.fit(df)

In [ ]:
# số quan sát có trong dữ liệu
df.shape[0] - 12

In [ ]:
# kiểm định chéo
df_cv = cross_validation(m,
                         horizon='90 days',
                         period='30 days',
                         initial='270 days',
                         parallel='processes')
df_cv.head()

In [ ]:
# hiệu suất
performance_metrics(df_cv).head()

In [ ]:
# RMSE và MAPE
print(f"RMSE: ", round(performance_metrics(df_cv)["rmse"].mean(), 1))
print(f"MAPE: ", 100 * round(performance_metrics(df_cv)["mape"].mean(), 3), "%")

In [ ]:
# sai số tăng theo thời gian?
plot_cross_validation_metric(df_cv, metric='rmse');

# Tinh chỉnh tham số

In [ ]:
# lưới tham số
param_grid = {'seasonality_mode':        ['additive', 'multiplicative'],
              'seasonality_prior_scale':  [1, 5, 10, 20],
              'holidays_prior_scale':     [5, 10, 20, 25],
              'changepoint_prior_scale':  [0.005, 0.01, 0.05, 0.1]}
grid = ParameterGrid(param_grid)
len(list(grid))

In [ ]:
# lưu kết quả
rmse = []

# vòng lặp
i = 1
for params in grid:
    print(f"{i} / {len(list(grid))}")
    # mô hình
    m = Prophet(holidays=holidays,
                seasonality_mode=params['seasonality_mode'],
                seasonality_prior_scale=params['seasonality_prior_scale'],
                holidays_prior_scale=params['holidays_prior_scale'],
                changepoint_prior_scale=params['changepoint_prior_scale'])
    m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    m.fit(df)
    # CV
    df_cv = cross_validation(m,
                             horizon='90 days',
                             period='30 days',
                             initial='270 days',
                             parallel='processes')
    # đo và lưu sai số
    error = performance_metrics(df_cv)["rmse"].mean()
    rmse.append(error)
    i += 1

In [ ]:
# xem kết quả
tuning_results = pd.DataFrame(grid)
tuning_results['rmse'] = rmse

# tìm chỉ số RMSE nhỏ nhất
min_rmse_index = tuning_results['rmse'].idxmin()

# bộ tham số tốt nhất
best_params = tuning_results.loc[min_rmse_index]
print("Bộ tham số tốt nhất (best_params):")
print(best_params)

In [ ]:
# xuất tham số tốt nhất
best_params.to_csv("../Forecasting Product/best_params_prophet.csv")